# CRED — SQL Queries

### Step 1 — Load the cleaned data

In [38]:
import pandas as pd
import sqlite3

raw = pd.read_excel('CRED_UPI_Data_Final.xlsx', sheet_name='Master Data')
raw.columns = ['month', 'app_name', 'volume', 'value']
raw.head()

,month,app_name,volume,value
0,Mar 2024,PhonePe,6502.15,1002387.84
1,Mar 2024,Google Pay,5061.18,695196.05
2,Mar 2024,Paytm,1230.04,134406.85
3,Mar 2024,Cred,132.31,44991.57
4,Mar 2024,Axis Bank Apps,94.60,7435.92


### Step 2 — Build the `apps` table
One row per unique app, with a stable ID.

In [39]:
unique_apps = sorted(raw['app_name'].unique())
apps = pd.DataFrame({
    'app_id': range(1, len(unique_apps) + 1),
    'app_name': unique_apps
})
print(f"{len(apps)} unique apps")
apps.head()

105 unique apps


,app_id,app_name
0,1,AU Small Finance Bank Apps
1,2,Aditya Birla Capital Digital
2,3,Airtel Payments Bank Apps
3,4,Allahabad Bank Apps
4,5,Amazon Pay


### Step 3 — Build the `monthly_app_stats` table

In [40]:
monthly_app_stats = raw.merge(apps, on='app_name', how='left')
monthly_app_stats = monthly_app_stats[['month', 'app_id', 'volume', 'value']]
monthly_app_stats.columns = ['month', 'app_id', 'txn_volume', 'txn_value']
monthly_app_stats.head()

,month,app_id,txn_volume,txn_value
0,Mar 2024,73,6502.15,1002387.84
1,Mar 2024,34,5061.18,695196.05
2,Mar 2024,71,1230.04,134406.85
3,Mar 2024,18,132.31,44991.57
4,Mar 2024,6,94.60,7435.92


### Step 4 — Push both tables into a real SQL database

In [41]:
conn = sqlite3.connect(':memory:')
apps.to_sql('apps', conn, index=False, if_exists='replace')
monthly_app_stats.to_sql('monthly_app_stats', conn, index=False, if_exists='replace')
print("Loaded: apps, monthly_app_stats")

Loaded: apps, monthly_app_stats


### Q1 Total volume/value by month

In [42]:
q1 = """
SELECT month, SUM(txn_volume) AS total_volume, SUM(txn_value) AS total_value
FROM monthly_app_stats
GROUP BY month
ORDER BY month
"""
pd.read_sql(q1, conn)

,month,total_volume,total_value
0,Apr 2024,13304.51,1964041.35
1,Apr 2025,17613.14,2374630.76
2,Aug 2024,14780.89,2048869.36
3,Aug 2025,19669.93,2458089.41
4,Dec 2024,16497.60,2307565.49
5,Dec 2025,21100.06,2764762.95
6,Feb 2025,15871.03,2179674.50
7,Feb 2026,19962.60,2653005.80
8,Jan 2025,16753.55,2330558.44
9,Jan 2026,21272.57,2800739.45


### Q2 Top-N apps by total volume, full window

In [43]:
q2 = """
SELECT a.app_name, SUM(m.txn_volume) AS total_volume_24mo
FROM monthly_app_stats m
JOIN apps a ON a.app_id = m.app_id
GROUP BY a.app_name
ORDER BY total_volume_24mo DESC
LIMIT 10
"""
pd.read_sql(q2, conn)

,app_name,total_volume_24mo
0,PhonePe,196056.94
1,Google Pay,149752.61
2,Paytm,30529.55
3,Navi,7524.91
4,super.money,3473.55
5,Cred,3450.44
6,Axis Bank Apps,2894.61
7,FamApp by Trio,2309.35
8,Amazon Pay,2032.22
9,BHIM,1639.18


### Q3 Market share per app per month (window function)

In [59]:
q3 = """
SELECT m.month, a.app_name, m.txn_volume,
       m.txn_volume * 100.0 / SUM(m.txn_volume) OVER (PARTITION BY m.month) AS market_share
FROM monthly_app_stats m
JOIN apps a ON a.app_id = m.app_id
ORDER BY m.month, market_share DESC
"""
shares = pd.read_sql(q3, conn)
shares.head(10)

,month,app_name,txn_volume,market_share
0,Apr 2024,PhonePe,6500.14,48.856666
1,Apr 2024,Google Pay,5027.32,37.786585
2,Apr 2024,Paytm,1117.13,8.396626
3,Apr 2024,Cred,138.46,1.040700
4,Apr 2024,Axis Bank Apps,69.80,0.524634
5,Apr 2024,Amazon Pay,64.33,0.483520
6,Apr 2024,ICICI Bank Apps,47.41,0.356345
7,Apr 2024,FamApp by Trio,46.64,0.350558
8,Apr 2024,Kotak Mahindra Bank Apps,41.54,0.312225
9,Apr 2024,HDFC Bank Apps,35.99,0.270510


### Q4 Month-on-month growth per app (window function)

In [45]:
q4 = """
SELECT a.app_name, m.month, m.txn_volume,
       LAG(m.txn_volume) OVER (PARTITION BY m.app_id ORDER BY m.month) AS prev_volume,
       (m.txn_volume - LAG(m.txn_volume) OVER (PARTITION BY m.app_id ORDER BY m.month)) * 1.0
         / LAG(m.txn_volume) OVER (PARTITION BY m.app_id ORDER BY m.month) AS mom_growth
FROM monthly_app_stats m
JOIN apps a ON a.app_id = m.app_id
WHERE a.app_name IN ('PhonePe', 'Google Pay', 'Paytm')
ORDER BY a.app_name, m.month
"""
pd.read_sql(q4, conn)

,app_name,month,txn_volume,prev_volume,mom_growth
0,Google Pay,Apr 2024,5027.32,NaN,NaN
1,Google Pay,Apr 2025,6488.55,5027.32,0.290658
2,Google Pay,Aug 2024,5592.45,6488.55,-0.138105
3,Google Pay,Aug 2025,7063.76,5592.45,0.263089
4,Google Pay,Dec 2024,6140.47,7063.76,-0.130708
...,...,...,...,...,...
67,PhonePe,Nov 2025,9322.29,7401.93,0.259440
68,PhonePe,Oct 2024,7905.54,9322.29,-0.151974
69,PhonePe,Oct 2025,9410.55,7905.54,0.190374
70,PhonePe,Sep 2024,7221.22,9410.55,-0.232646


### Q5 Rank apps within each month (window function)

In [46]:
q5 = """
SELECT m.month, a.app_name, m.txn_volume,
       RANK() OVER (PARTITION BY m.month ORDER BY m.txn_volume DESC) AS rank_in_month
FROM monthly_app_stats m
JOIN apps a ON a.app_id = m.app_id
ORDER BY m.month, rank_in_month
"""
ranked = pd.read_sql(q5, conn)
ranked.head(10)

,month,app_name,txn_volume,rank_in_month
0,Apr 2024,PhonePe,6500.14,1
1,Apr 2024,Google Pay,5027.32,2
2,Apr 2024,Paytm,1117.13,3
3,Apr 2024,Cred,138.46,4
4,Apr 2024,Axis Bank Apps,69.80,5
5,Apr 2024,Amazon Pay,64.33,6
6,Apr 2024,ICICI Bank Apps,47.41,7
7,Apr 2024,FamApp by Trio,46.64,8
8,Apr 2024,Kotak Mahindra Bank Apps,41.54,9
9,Apr 2024,HDFC Bank Apps,35.99,10


### Q6 Concentration (HHI) via a CTE

In [47]:
q6 = """
WITH shares AS (
  SELECT m.month, m.app_id,
         m.txn_volume * 1.0 / SUM(m.txn_volume) OVER (PARTITION BY m.month) AS share
  FROM monthly_app_stats m
)
SELECT month, SUM(share * share) AS hhi
FROM shares
GROUP BY month
ORDER BY month
"""
hhi = pd.read_sql(q6, conn)
hhi

,month,hhi
0,Apr 2024,0.388751
1,Apr 2025,0.366571
2,Aug 2024,0.388458
3,Aug 2025,0.351707
4,Dec 2024,0.378049
5,Dec 2025,0.343204
6,Feb 2025,0.373189
7,Feb 2026,0.339217
8,Jan 2025,0.375378
9,Jan 2026,0.340405


### Q7 3 MONTHS MOVING AVERAGE

In [48]:
q7 = """
SELECT month, total_volume,
       AVG(total_volume) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_avg_3mo
FROM (
  SELECT month, SUM(txn_volume) AS total_volume
  FROM monthly_app_stats
  GROUP BY month
)
ORDER BY month
"""
moving_avg = pd.read_sql(q7, conn)
moving_avg

,month,total_volume,moving_avg_3mo
0,Apr 2024,13304.51,13304.510000
1,Apr 2025,17613.14,15458.825000
2,Aug 2024,14780.89,15232.846667
3,Aug 2025,19669.93,17354.653333
4,Dec 2024,16497.60,16982.806667
5,Dec 2025,21100.06,19089.196667
6,Feb 2025,15871.03,17822.896667
7,Feb 2026,19962.60,18977.896667
8,Jan 2025,16753.55,17529.060000
9,Jan 2026,21272.57,19329.573333


### Q8 YEAR ON YEAR GROWTH

In [49]:
q8 = """
WITH ranked AS (
  SELECT m.*, a.app_name,
         ROW_NUMBER() OVER (PARTITION BY m.app_id ORDER BY m.month) AS rn
  FROM monthly_app_stats m
  JOIN apps a ON a.app_id = m.app_id
)
SELECT curr.app_name, curr.month, curr.txn_volume,
       prior.txn_volume AS volume_12mo_ago,
       (curr.txn_volume - prior.txn_volume) * 1.0 / prior.txn_volume AS yoy_growth
FROM ranked curr
JOIN ranked prior ON curr.app_id = prior.app_id AND curr.rn = prior.rn + 12
ORDER BY curr.app_name, curr.month
"""
yoy_growth = pd.read_sql(q8, conn)
yoy_growth

,app_name,month,txn_volume,volume_12mo_ago,yoy_growth
0,AU Small Finance Bank Apps,Jun 2024,0.76,0.80,-0.050000
1,AU Small Finance Bank Apps,Jun 2025,0.62,0.57,0.087719
2,AU Small Finance Bank Apps,Mar 2024,0.78,0.53,0.471698
3,AU Small Finance Bank Apps,Mar 2025,0.59,0.68,-0.132353
4,AU Small Finance Bank Apps,May 2024,0.82,0.49,0.673469
...,...,...,...,...,...
827,super.money,Nov 2025,265.07,286.86,-0.075960
828,super.money,Oct 2024,49.68,139.10,-0.642847
829,super.money,Oct 2025,264.77,289.32,-0.084854
830,super.money,Sep 2024,25.75,124.83,-0.793719


### Q9 RANK STABILITY OF APPS

In [50]:
q9 = """
WITH ranked AS (
  SELECT m.month, a.app_name,
         RANK() OVER (PARTITION BY m.month ORDER BY m.txn_volume DESC) AS rank_in_month
  FROM monthly_app_stats m
  JOIN apps a ON a.app_id = m.app_id
)
SELECT app_name, COUNT(*) AS months_in_top10
FROM ranked
WHERE rank_in_month <= 10
GROUP BY app_name
ORDER BY months_in_top10 DESC
"""
top10_apps = pd.read_sql(q9, conn)
top10_apps

,app_name,months_in_top10
0,PhonePe,24
1,Paytm,24
2,Google Pay,24
3,FamApp by Trio,24
4,Cred,24
5,Axis Bank Apps,24
6,Navi,20
7,Amazon Pay,20
8,super.money,16
9,WhatsApp,12


### Q10 CRED'S BEST MONTH

In [51]:
q10 = """
SELECT m.month, m.txn_volume, m.txn_value
FROM monthly_app_stats m
JOIN apps a ON a.app_id = m.app_id
WHERE a.app_name = 'Cred'
ORDER BY m.txn_volume DESC
LIMIT 1
"""
cred_top_month = pd.read_sql(q10, conn)
cred_top_month

,month,txn_volume,txn_value
0,Oct 2025,157.99,62438.03


### Extensibility self-check
Could this take a 25th month of new data and rerun cleanly with no manual edits? Write your own reasoning here for the write-up.

In [55]:
print("Structural check: every query above keys off app_id / month only, "
      "so adding a 25th month means re-running Steps 1-4 with the new rows, "
      "then every query cell unchanged.")

Structural check: every query above keys off app_id / month only, so adding a 25th month means re-running Steps 1-4 with the new rows, then every query cell unchanged.


### Export for teammates

In [54]:
shares.to_csv('cred_market_shares.csv', index=False)
hhi.to_csv('cred_concentration_hhi.csv', index=False)
ranked.to_csv('cred_app_rankings.csv', index=False)
print("Saved: cred_market_shares.csv, cred_concentration_hhi.csv, cred_app_rankings.csv")

Saved: cred_market_shares.csv, cred_concentration_hhi.csv, cred_app_rankings.csv
